In [1]:

from pathlib import Path
import joblib
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Find the project root
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

MODELS_DIR = PROJECT_ROOT / "models"

# Sentiment labels
LABELS = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

# Load Logistic Regression and TF-IDF
logistic_model = joblib.load(
    MODELS_DIR / "logistic_regression_model.pkl"
)

tfidf_vectorizer = joblib.load(
    MODELS_DIR / "tfidf_vectorizer.pkl"
)

# Load final selected DistilBERT checkpoint
distilbert_path = MODELS_DIR / "distilbert_full_epoch2"

tokenizer = AutoTokenizer.from_pretrained(
    str(distilbert_path),
    local_files_only=True
)

distilbert_model = AutoModelForSequenceClassification.from_pretrained(
    str(distilbert_path),
    local_files_only=True
)

distilbert_model.to("cpu")
distilbert_model.eval()

print("Both saved models loaded successfully!")
print("Device: CPU")


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Both saved models loaded successfully!
Device: CPU


In [2]:

import pandas as pd

# Example Amazon Beauty product reviews
sample_reviews = [
    "This product is amazing! I absolutely love it.",
    "Terrible quality. It broke after one use.",
    "The product is okay, but nothing special.",
    "Excellent quality and fast delivery.",
    "The packaging was nice, but the product did not work well."
]

def predict_logistic(text):
    features = tfidf_vectorizer.transform([text])
    prediction = logistic_model.predict(features)[0]
    return LABELS[int(prediction)]

def predict_distilbert(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = distilbert_model(**inputs)
        prediction = torch.argmax(
            outputs.logits, dim=1
        ).item()

    return LABELS[prediction]

# Compare predictions
results = []

for review in sample_reviews:
    logistic_prediction = predict_logistic(review)
    distilbert_prediction = predict_distilbert(review)

    results.append({
        "Review": review,
        "Logistic Regression": logistic_prediction,
        "DistilBERT": distilbert_prediction
    })

demo_df = pd.DataFrame(results)

display(demo_df)


,Review,Logistic Regression,DistilBERT
0,This product is amazing! I absolutely love it.,Positive,Positive
1,Terrible quality. It broke after one use.,Negative,Negative
2,"The product is okay, but nothing special.",Negative,Neutral
3,Excellent quality and fast delivery.,Positive,Positive
4,"The packaging was nice, but the product did no...",Negative,Negative


In [3]:

# Save model demonstration results

output_path = PROJECT_ROOT / "data" / "model_demo_predictions.csv"

demo_df.to_csv(output_path, index=False)

print("Demonstration results saved successfully!")
print("File:", output_path)
print("Total reviews tested:", len(demo_df))


Demonstration results saved successfully!
File: c:\Users\jyothi\sentiment-analysis-nlp\data\model_demo_predictions.csv
Total reviews tested: 5
